# CS5202 — Financial RAG: Data Exploration
Explore the Indian Financial News dataset before building the pipeline.

In [ ]:
import sys, json, random
sys.path.insert(0, '../src')
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
SEED = 42
random.seed(SEED)

## 1. Load dataset

In [ ]:
with open('../data/financial_news.json') as f:
    articles = json.load(f)
print(f'Total articles: {len(articles)}')
print(f'Sample: {articles[0]}')

## 2. Article length distribution

In [ ]:
lengths = [len(a['text'].split()) for a in articles]
df = pd.DataFrame({'word_count': lengths})
print(df.describe())
plt.hist(lengths, bins=40, color='steelblue', edgecolor='white')
plt.xlabel('Article length (words)')
plt.ylabel('Count')
plt.title('Article Length Distribution — Indian Financial News (26k)')
plt.tight_layout()
plt.savefig('../results/exploration_article_lengths.png', dpi=120)
plt.show()

## 3. Source distribution

In [ ]:
sources = Counter(a.get('source','unknown') for a in articles)
top = sources.most_common(10)
print('Top 10 sources:')
for src, cnt in top:
    print(f'  {src}: {cnt}')

## 4. Keyword frequency — financial topics

In [ ]:
keywords = ['RBI', 'SEBI', 'repo rate', 'inflation', 'GDP', 'Nifty',
            'earnings', 'IPO', 'merger', 'rupee', 'FII', 'bank']
kw_counts = {}
all_text = ' '.join(a['text'].lower() for a in articles)
for kw in keywords:
    kw_counts[kw] = all_text.count(kw.lower())
kw_df = pd.DataFrame(list(kw_counts.items()), columns=['keyword','count']).sort_values('count', ascending=False)
print(kw_df)
kw_df.plot.bar(x='keyword', y='count', legend=False, color='steelblue')
plt.title('Financial Keyword Frequency in Dataset')
plt.ylabel('Occurrences')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('../results/exploration_keyword_freq.png', dpi=120)
plt.show()

## 5. Quick retrieval spot-check

In [ ]:
from retriever import build_index, retrieve
index, chunks = build_index(articles, chunk_size=250, overlap=40)
query = 'What is the current RBI repo rate?'
results, scores = retrieve(query, index, chunks, top_k=3)
print(f'Query: {query}')
for i, (c, s) in enumerate(zip(results, scores), 1):
    print(f'\n[{i}] score={s:.4f} | {c["title"]}')
    print(c['text'][:200], '...')